# Notebook13 — runner
Purpose: registered full JEPA product-geometry ablation. Imports execute explicitly tagged definitions only.
Inputs: fixed Food101 training/development membership; official test remains closed. Outputs: artifacts/notebook_13/<run_id>/ and checkpoints/<run_id>/.
Expected runtime: definition import seconds; actual study12h or registered3h cap. Resume: unchanged notebooks/13_Product_Geometry_JEPA.ipynb, launch-product-geometry cell, before original deadline.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT=Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot={"__file__":str(ROOT/"fmm_jepa/bootstrap.ipynb")}
for _cell in nbformat.read(ROOT/"fmm_jepa/bootstrap.ipynb",as_version=4).cells:
    if _cell.cell_type=="code" and "ahr-export" in _cell.metadata.get("tags",[]):
        exec(compile(_cell.source,"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](ROOT)

"""Notebook13 source-bound overnight variant training; importing definitions never trains."""
from pathlib import Path
import copy, gc, hashlib, json, os, sys, time, types
import torch
from fmm_jepa.training import atomic_json, digest_file, definition_hashes
from fmm_jepa.food101_training import isolated_rng, _hash
import fmm_jepa.semantic_pilot as _pilot

ROOT = Path(__file__).resolve().parents[2]
RESUME_NOTEBOOK = 'notebooks/13_Product_Geometry_JEPA.ipynb'
RESUME_CELL = 'launch-product-geometry'

def load_tagged(root, relative, name):
    """Load only the explicit reusable definition cells, once per process."""
    path=Path(root)/relative
    if name in sys.modules:
        module=sys.modules[name]
        if Path(module.__file__).resolve()!=path.resolve():raise ValueError('Module source path changed')
        return module
    module=types.ModuleType(name);module.__file__=str(path.resolve());sys.modules[name]=module
    cells=json.loads(path.read_text())['cells']
    selected=[c for c in cells if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])]
    if not selected:raise ValueError('Notebook has no reusable definition cells')
    try:
        for index,cell in enumerate(selected):
            exec(compile(''.join(cell['source']),str(path)+'#ahr-export-'+str(index),'exec'),module.__dict__)
    except BaseException:
        sys.modules.pop(name,None);raise
    return module

_BASE=load_tagged(ROOT,'experiments/05_scaling/runner.ipynb','fmm13_inherited_runner')

# Inherited functions resolve within an isolated module namespace. Upstream
# notebook files and canonical Notebook05 modules remain unchanged.
normalize_stage_plan=_BASE.normalize_stage_plan

def stage_definition_hashes(root):
    result=_BASE_ORIGINAL_HASHES(root)
    for folder in ('experiments/07_energy','experiments/08_replication','experiments/09_deep_energy','experiments/10_manifold_kernel','experiments/11_manifold_long','experiments/13_product_geometry'):
        for path in sorted((Path(root)/folder).glob('*.ipynb')):
            cells=json.loads(path.read_text())['cells']
            sources=[''.join(c['source']) for c in cells if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])]
            result[str(path.relative_to(root))]=hashlib.sha256('\n'.join(sources).encode()).hexdigest()
    return result

_BASE_ORIGINAL_HASHES=_BASE.stage_definition_hashes

def configure_input_pipeline(config):
    workers=int(config.get('input_pipeline',{}).get('workers',0))
    if workers<0:raise ValueError('Nonnegative ordered input workers required')
    if workers:
        pipeline=load_tagged(ROOT,'experiments/11_manifold_long/input_pipeline.ipynb','fmm11_input_pipeline')
        return pipeline.install_input_pipeline(workers=workers)
    return {'status':'SERIAL','workers':0}

def build_model(config,kind):
    configure_input_pipeline(config)
    if config.get('execution_optimization',False):
        engineering=load_tagged(ROOT,'experiments/11_manifold_long/engineering.ipynb','fmm11_engineering')
        engineering.install_execution_optimizations(ROOT)
    options=copy.deepcopy(config.get('kernel13',{}));variant=options.get('variant')
    if kind=='vit':
        if variant not in (None,'vit'):raise ValueError('ViT arm has a different declared variant')
        return _pilot.build_model(config,'vit')
    if kind!='semantic' or variant not in ('vit','learned_kernel','normalized_kernel','product_kernel','sphere_kernel','multi_view_kernel'):
        raise ValueError('Explicit registered kernel13 variant required')
    backend=load_tagged(ROOT,'experiments/13_product_geometry/encoder.ipynb','fmm13_encoder')
    model_config=copy.deepcopy(config['model']);model_config['kernel13']=options
    return backend.build_model(model_config,kind='semantic',variant=variant)

def initialize_models(config,artifact):
    """Matched scaffold, values and comparable geometry maps; independent storage."""
    from fmm_jepa.food101_evaluation import encoder_state_hash
    variant=config.get('kernel13',{}).get('variant')
    variants=('vit','learned_kernel','normalized_kernel','product_kernel','sphere_kernel','multi_view_kernel')
    if variant not in variants:raise ValueError('Explicit kernel13 variant required')
    folder=Path(artifact)/'initialization';folder.mkdir(parents=True,exist_ok=True);path=folder/'manifest.json'
    identity=_hash({'model':config['model'],'kernel13':config['kernel13'],'seed':config['seeds']['model_init'],
        'initialization_policy':'shared-scaffold-values-bandwidths-mixture-base-coordinate-maps/v2'})
    baseline=copy.deepcopy(config);baseline['kernel13']['variant']='vit'
    with isolated_rng(config['seeds']['model_init']):
        models={'vit':build_model(baseline,'vit').cpu()};reference=None
        if variant!='vit':
            reference_config=copy.deepcopy(config);reference_config['kernel13']['variant']='learned_kernel'
            reference=build_model(reference_config,'semantic').cpu()
            models['semantic']=reference if variant=='learned_kernel' else build_model(config,'semantic').cpu()
    if path.exists():
        info=json.loads(path.read_text())
        if info['identity']!=identity or set(info['states'])!=set(models):raise ValueError('Initialization identity/architectures differ')
        for kind,model in models.items():
            for record in info['states'][kind].values():
                if digest_file(record['path'])!=record['sha256']:raise ValueError('Initial-state checksum mismatch')
            model.load_state_dict(torch.load(info['states'][kind]['model']['path'],map_location='cpu',weights_only=True))
        return models,info
    if list(folder.glob('*.pt')):raise ValueError('Orphan initialization states; refusing overwrite')
    source=models['vit'];source_state=source.context.state_dict()
    def common(name):
        parts=name.split('.')
        return name.startswith(('patch_embed.','norm.')) or (len(parts)>=4 and parts[0]=='blocks' and parts[2] in ('norm1','norm2','mlp'))
    context_names=[name for name in source_state if common(name)]
    if {int(name.split('.')[1]) for name in context_names if name.startswith('blocks.')}!=set(range(int(config['model']['depth']))):
        raise AssertionError('Common scaffold must cover every block')
    copied=['context.'+name for name in context_names]+['predictor.'+name for name in source.predictor.state_dict()]
    mixer_names=[];mixer_hash=None;geometry_copied=[]
    if variant!='vit':
        destination=models['semantic'];dest_state=destination.context.state_dict()
        for name in context_names:
            if name not in dest_state or dest_state[name].shape!=source_state[name].shape:raise AssertionError('Common tensor mismatch: '+name)
            dest_state[name].copy_(source_state[name])
        destination.context.load_state_dict(dest_state);destination.predictor.load_state_dict(source.predictor.state_dict())
        for name in copied:
            a,b=source.state_dict()[name],destination.state_dict()[name]
            if not torch.equal(a,b) or a.data_ptr()==b.data_ptr():raise AssertionError('Common tensor values/storage mismatch')
        if variant!='learned_kernel':
            backend=load_tagged(ROOT,'experiments/13_product_geometry/encoder.ipynb','fmm13_encoder')
            for index,(dest_block,ref_block) in enumerate(zip(destination.context.blocks,reference.context.blocks)):
                names=backend.initialize_mixer_from_reference(dest_block.mixer,ref_block.mixer)
                geometry_copied.extend('blocks.'+str(index)+'.mixer.'+name for name in names)
        reference_state=reference.context.state_dict();destination_state=destination.context.state_dict()
        shared_suffixes=('value.weight','value.bias','projection.weight','projection.bias','log_bandwidths','mixture_logits')
        mixer_names=[name for name in reference_state if '.mixer.' in name and name.split('.mixer.',1)[1] in shared_suffixes]
        if len(mixer_names)!=6*int(config['model']['depth']):raise AssertionError('Every kernel block requires six matched mixer tensors')
        for name in mixer_names:
            a,b=reference_state[name],destination_state[name]
            if not torch.equal(a,b):raise AssertionError('Kernel reference tensor mismatch: '+name)
            if variant!='learned_kernel' and a.data_ptr()==b.data_ptr():raise AssertionError('Independent kernel storage required')
        mixer_hash=encoder_state_hash({name:destination_state[name].detach().cpu() for name in mixer_names})
    records={};counts={};shared={};inactive={}
    for kind,model in models.items():
        model.target.load_state_dict(model.context.state_dict());model.target.requires_grad_(False).eval()
        counts[kind]=_pilot.parameter_counts(model);records[kind]={}
        shared[kind]=encoder_state_hash({name:model.state_dict()[name].detach().cpu() for name in copied});inactive[kind]=0
        for name,state in [('model',model.state_dict()),('teacher',model.target.state_dict())]:
            file=folder/f'{kind}_initial_{name}.pt';_atomic_torch(file,state);records[kind][name]=_record(file)
    if len(set(shared.values()))!=1:raise AssertionError('Common scaffold hashes differ')
    info={'identity':identity,'variant':variant,'states':records,'parameter_counts':counts,'copied_components':copied,
        'shared_parameter_hashes':shared,'kernel_mixer_shared_sha256':mixer_hash,'kernel_mixer_shared_names':mixer_names,
        'geometry_reference_copied_names':geometry_copied,'inactive_context_scaffold_parameters':inactive,
        'independent_storage_verified':True,'comparison':'Shared stem, all block MLP/norms, final norm, predictor and kernel value/output/bandwidth/mixture; comparable coordinate maps copied from original learned reference. Extra geometry parameters active and reported; architecture behavior may differ at initialization.',
        'snapshot_purpose':'Immutable initialization controls, not resumable training checkpoints'}
    atomic_json(path,info);atomic_json(Path(artifact)/'initialization_manifest.json',info)
    return models,info

for _name in ('_strict_training_protocol',):
    globals()[_name]=getattr(_BASE,_name)

def _plan_binding(root,plan,bundle):
    safe=lambda s:isinstance(s,str) and bool(s) and all(c.isalnum() or c in '-_' for c in s)
    if not safe(plan.get('run_id')) or not safe(plan.get('stage_id')):raise ValueError('Safe run_id and stage_id required')
    arms=plan.get('arms',[])
    if not arms or len({a['id'] for a in arms})!=len(arms) or any(not safe(a['id']) for a in arms):
        raise ValueError('Unique safe arm IDs required')
    updates=int(plan['updates']);milestones=list(map(int,plan.get('milestones',[updates])))
    if updates<4 or len(set(milestones))!=len(milestones) or any(m<1 or m>updates for m in milestones):
        raise ValueError('Invalid schedule or milestone set')
    if updates not in milestones:raise ValueError('Final update must be an explicit milestone')
    if int(plan.get('round_updates',100))<1:raise ValueError('Positive round size required')
    train=list(bundle['train_ids']);probe=list(bundle['probe_train_ids']);dev=list(bundle['dev_ids']);heldout=list(bundle.get('heldout_ids',[]))
    if not train or not probe or not dev or any(len(ids)!=len(set(ids)) for ids in (train,probe,dev)):
        raise ValueError('Nonempty unique training/probe/development memberships required')
    if set(train)&set(dev) or set(probe)&set(dev) or (set(train)|set(probe)|set(dev))&set(heldout):
        raise ValueError('Development or official-test leakage')
    if not set(probe).issubset(train):raise ValueError('Fixed probe training set must be nested inside stage pretraining pool')
    if 'train_count' in plan and int(plan['train_count'])!=len(train):raise ValueError('Stage training count differs from plan')
    if 'probe_train_count' in plan and int(plan['probe_train_count'])!=len(probe):raise ValueError('Probe count differs from plan')
    seeds=sorted({int(a['seed']) for a in arms})
    if 'seeds' in plan and sorted(map(int,plan['seeds']))!=seeds:raise ValueError('Declared seed list differs from arms')
    for arm in arms:
        config=arm['config'];policy=_strict_training_protocol(config)
        if arm['kind'] not in ('vit','semantic') or int(config['seeds']['model_init'])!=int(arm['seed']):
            raise ValueError('Architecture or model seed mismatch')
        if policy['updates']!=updates or policy['effective_batch']!=int(plan.get('effective_batch',128)):
            raise ValueError('Schedule/effective batch differs from registered stage')
        if config.get('dtype','float32')!='float32':raise ValueError('Only verified FP32 enabled')
        if config.get('evaluation',{}).get('heldout_enabled',False):raise ValueError('Official test is reserved')
        if int(config['probe']['epochs'])!=20:raise ValueError('Fixed epoch20 probes required')
        if policy['checkpoint_every']>50:raise ValueError('Checkpoint at most every50 successful updates')
    variants=plan.get('variants',[])
    budget=int(plan.get('hard_limit_seconds',43200))
    expected_seeds=list(range(20261501,20261509 if budget==43200 else 20261507))
    candidates=(1000,800,600) if budget==43200 else (600,350,200)
    if budget not in (43200,10800) or seeds!=expected_seeds:raise ValueError('Registered budget and fresh paired seeds required')
    if updates not in candidates or milestones!=[updates]:raise ValueError('Only timing-selected final update milestone allowed')
    if tuple(variants)!=('vit','learned_kernel','normalized_kernel','product_kernel','sphere_kernel','multi_view_kernel'):
        raise ValueError('Exactly six ordered registered variants required')
    if plan.get('probe_endpoints')!=(['initial','final'] if budget==43200 else ['final']):raise ValueError('Probe endpoints differ from registered budget design')
    for seed in seeds:
        group=[a for a in arms if int(a['seed'])==seed]
        if sorted(a['variant'] for a in group)!=sorted(variants):raise ValueError('Every seed must contain every registered variant exactly once')
        configs=[]
        for arm in group:
            config=normalize_stage_plan(arm['config'])
            options=config.pop('kernel13',None)
            if not options or options.get('variant')!=arm['variant']:raise ValueError('Declared kernel13 variant differs from arm')
            if arm['variant']=='vit':
                if arm['kind']!='vit':raise ValueError('ViT variant has wrong architecture')
            elif arm['kind']!='semantic' or not options or options['variant']!=arm['variant']:
                raise ValueError('Kernel encoder variant/config mismatch')
            if not options:raise ValueError('Explicit kernel13 configuration required for every arm')
            shared_options={key:value for key,value in options.items() if key!='variant'}
            config['kernel13_common_options']=shared_options
            config.get('model',{}).pop('kind',None)
            configs.append(config)
        if any(config!=configs[0] for config in configs[1:]):
            raise ValueError('Paired variants differ beyond architecture and registered kernel13 variant choice only')
    sources=plan.get('source_hashes',stage_definition_hashes(root));actual=stage_definition_hashes(root)
    if sources!=actual:raise ValueError('Registered Notebook05 definition hashes changed')
    consumer_hash=plan.get('consumer_notebook_sha256')
    if consumer_hash:
        nb=json.loads((Path(root)/plan.get('resume_notebook',RESUME_NOTEBOOK)).read_text())
        code=[''.join(c['source']) for c in nb['cells'] if c['cell_type']=='code']
        if _hash(code)!=consumer_hash:raise ValueError('Registered main notebook code changed')
    return {'schema':'fmm-jepa-notebook13-stage-binding/v1','plan_hash':_hash(plan),
        'source_hashes':sources,'dataset_fingerprint':bundle['dataset'].fingerprint,
        'split_sha256':bundle['split_manifest']['split_sha256'],
        'training_ids_sha256':_hash(train),'training_examples':len(train),
        'probe_training_ids_sha256':_hash(probe),'probe_training_examples':len(probe),
        'development_ids_sha256':_hash(dev),'development_examples':len(dev),'heldout_status':'NOT_RUN'}

def _persist_numerical_backend_stats(artifact,stage_id,arm_id,event):
    backend=sys.modules.get('fmm13_encoder')
    value={'status':'RECORDED','stage_id':stage_id,'arm_id':arm_id,'event':event,
        'updated_unix':time.time(),'worker_pid':os.getpid(),
        'counter_scope':'Current process cumulative, including calibration/preflight and all earlier arms; not checkpoint state or training-only performance.'}
    for name in ('get_stats','stats'):
        if backend is not None and callable(getattr(backend,name,None)):
            value['statistics']=getattr(backend,name)();break
    atomic_json(Path(artifact)/'numerical_backend_stats.json',value)
    return value

for _name in ('build_model','initialize_models','stage_definition_hashes','_plan_binding','_persist_numerical_backend_stats'):
    setattr(_BASE,_name,globals()[_name])
_BASE.RESUME_NOTEBOOK=RESUME_NOTEBOOK
_BASE.RESUME_CELL=RESUME_CELL
# Make the inherited globals available to the deliberately adapted preflight.
for _name in ('_rng','_restore_rng','_atomic_torch','_record','_guard','_deadline','_status',
              '_registered','_arm_paths','_initial_model','_run_stage_model','_release_semantic','validate_training_manifest'):
    globals()[_name]=getattr(_BASE,_name)
def verify_stage_resume(root,artifact,plan,bundle,deadline_unix=None):
    """At user execution only: actual-config four updates equals reconstructed2+2.

    One first-seed certificate per architecture covers that stage resolution,
    batch/accumulation, data membership and full successful-update schedule.
    Every RNG/optimizer/model/scheduler/EMA/sampler/trace is compared bitwise;
    elapsed timing is the only excluded state.
    """
    from fmm_jepa.food101_benchmark import _food_exact_tree
    root,artifact=Path(root),Path(artifact);plan=normalize_stage_plan(plan)
    registration=_registered(root,artifact,plan,bundle);path=artifact/'stage_preflight.json'
    if path.exists():
        certificate=json.loads(path.read_text())
        if certificate['status']!='PASS' or certificate['binding']!=registration['binding']:
            raise ValueError('Actual-config preflight identity differs')
        expected={next(a['id'] for a in plan['arms'] if a['variant']==variant) for variant in plan['variants']}
        if set(certificate['arms'])!=expected:raise ValueError('Resume certificate must cover every registered variant')
        for entry in certificate['arms'].values():
            for record in entry['checkpoints'].values():
                folder=Path(record['directory']);manifest=validate_training_manifest(folder)
                if manifest['successful_updates']!=4 or digest_file(folder/manifest['latest'])!=record['sha256']:
                    raise ValueError('Full-state preflight evidence changed')
        return certificate
    selected=[]
    for variant in plan['variants']:
        selected.append(next(a for a in plan['arms'] if a['variant']==variant))
    fields=['model','optimizer','scheduler','ema_position','scaler','precision','rng','sampler',
            'streams','microstep','partial_accumulation','world_size','rank','replay','config','protocol','identity']
    state_fields=['successful_updates','attempted_updates','image_exposures','consecutive_failures','traces','failures','diagnostics']
    timing={'data_seconds','compute_seconds','step_seconds','images_per_second','_cuda_event_count'}
    certificate={'status':'RUNNING','binding':registration['binding'],'arms':{},
        'scope':'First seed per registered variant; exact actual stage config/data and full schedule; elapsed timing excluded'}
    original_rng=_rng()
    try:
        with _guard(deadline_unix):
            for arm in selected:
                _deadline(deadline_unix,5);_status(artifact,'exact_resume','RUNNING',arm_id=arm['id'])
                arm_folder,_=_arm_paths(root,artifact,plan,arm)
                protocol=json.loads((arm_folder/'protocol.json').read_text())
                folder=artifact/'preflight'/arm['id'];folder.mkdir(parents=True,exist_ok=True)
                rng_path=folder/'trajectory_rng.pt';rng_record=folder/'trajectory_rng.json'
                if rng_record.exists():
                    if digest_file(rng_path)!=json.loads(rng_record.read_text())['sha256']:raise ValueError('Preflight RNG changed')
                    trajectory=torch.load(rng_path,map_location='cpu',weights_only=False)
                else:
                    if rng_path.exists():raise ValueError('Orphan trajectory RNG; inspect before recovery')
                    with isolated_rng(int(arm['seed'])):trajectory=_rng()
                    _atomic_torch(rng_path,trajectory);atomic_json(rng_record,_record(rng_path))
                reference_dir=root/'checkpoints'/plan['run_id']/'stages'/plan['stage_id']/'preflight'/(arm['id']+'-reference')
                split_dir=reference_dir.with_name(arm['id']+'-split')
                _restore_rng(copy.deepcopy(trajectory))
                reference=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'reference',reference_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=4,deadline_unix=deadline_unix)
                del reference
                _restore_rng(copy.deepcopy(trajectory))
                partial=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'split',split_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=2,deadline_unix=deadline_unix)
                if partial['resume']['final_step'] not in (2,4):raise AssertionError('Preflight cutoff before split boundary')
                del partial
                resumed=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'split',split_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=4,deadline_unix=deadline_unix)
                lm,rm=validate_training_manifest(reference_dir),validate_training_manifest(split_dir)
                left=torch.load(reference_dir/lm['latest'],map_location='cpu',weights_only=False)
                right=torch.load(split_dir/rm['latest'],map_location='cpu',weights_only=False)
                for field in fields:_food_exact_tree(left[field],right[field],field)
                for field in state_fields:_food_exact_tree(left['state'][field],right['state'][field],'state.'+field)
                lmetrics=[{k:v for k,v in row.items() if k not in timing} for row in left['state']['metrics']]
                rmetrics=[{k:v for k,v in row.items() if k not in timing} for row in right['state']['metrics']]
                _food_exact_tree(lmetrics,rmetrics,'deterministic_metrics')
                if lm['successful_updates']!=4 or rm['attempted_updates']!=4 or len(rm['checkpoints'])<2:
                    raise AssertionError('Four finite updates and latest-two retention required')
                certificate['arms'][arm['id']]={'status':'PASS','bitwise_equal':True,
                    'image_size':arm['config']['image_size'],'training_examples':len(bundle['train_ids']),
                    'microbatch':arm['config']['training']['microbatch'],'accumulation_steps':arm['config']['training']['accumulation_steps'],
                    'schedule_updates':plan['updates'],'full_state_fields':fields,
                    'checkpoints':{'reference':{'directory':str(reference_dir),'sha256':digest_file(reference_dir/lm['latest'])},
                                   'resumed':{'directory':str(split_dir),'sha256':digest_file(split_dir/rm['latest'])}}}
                atomic_json(folder/'certificate.json',certificate['arms'][arm['id']])
                del left,right,resumed;_release_semantic(arm['config']['device'])
    finally:_restore_rng(original_rng)
    certificate['status']='PASS';atomic_json(path,certificate)
    return certificate
_BASE.verify_stage_resume=verify_stage_resume
def prepare_stage(root,artifact,plan,bundle):
    from fmm_jepa.food101_evaluation import encoder_state_hash
    root,artifact=Path(root),Path(artifact);plan=normalize_stage_plan(plan)
    artifact.mkdir(parents=True,exist_ok=True);binding=_plan_binding(root,plan,bundle)
    path=artifact/'stage_registration.json'
    if path.exists():
        registration=json.loads(path.read_text())
        if registration['binding']!=binding or registration['normalized_plan']!=plan:raise ValueError('Existing registration differs; no silent restart')
    else:registration={'binding':binding,'normalized_plan':plan,'arms':{},'heldout_status':'NOT_RUN'}
    paired_shared={};paired_kernel_mixer={}
    for arm in plan['arms']:
        folder,checkpoint=_arm_paths(root,artifact,plan,arm);folder.mkdir(parents=True,exist_ok=True)
        models,initial=initialize_models(arm['config'],folder);model=models[arm['kind']]
        shared=initial['shared_parameter_hashes'][arm['kind']];seed=int(arm['seed'])
        if seed in paired_shared and paired_shared[seed]!=shared:raise AssertionError('Paired common initial parameters differ')
        paired_shared[seed]=shared
        mixer_hash=initial.get('kernel_mixer_shared_sha256')
        if mixer_hash is not None:
            if seed in paired_kernel_mixer and paired_kernel_mixer[seed]!=mixer_hash:raise AssertionError('Paired kernel value/output/bandwidth/mixture initialization differs')
            paired_kernel_mixer[seed]=mixer_hash
        parameter_hash=encoder_state_hash({name:value.detach().cpu() for name,value in model.named_parameters() if value.requires_grad})
        protocol={'schema':'fmm-jepa-notebook13-arm/v1','binding':binding,'source_hashes':binding['source_hashes'],
            'arm':arm,'initialization':initial,'training':_strict_training_protocol(arm['config']),
            'milestones':plan['milestones'],'selection':'Predeclared final full-state checkpoint; fixed probe epoch20',
            'heldout_status':'NOT_RUN'}
        protocol_path=folder/'protocol.json'
        if protocol_path.exists() and json.loads(protocol_path.read_text())!=protocol:raise ValueError('Arm protocol changed')
        atomic_json(protocol_path,protocol)
        registration['arms'][arm['id']]={'protocol_sha256':digest_file(protocol_path),'initialization':initial,
            'initial_teacher':initial['states'][arm['kind']]['teacher'],'trainable_parameter_hash':parameter_hash,
            'shared_parameter_hash':shared,'kernel_mixer_shared_sha256':mixer_hash,'variant':arm['variant'],
            'checkpoint_directory':str(checkpoint.resolve())}
        del model,models;gc.collect()
    atomic_json(path,registration)
    return registration
_BASE.prepare_stage=prepare_stage
train_stage=_BASE.train_stage
validate_stage_milestones=_BASE.validate_stage_milestones

def calibrate_case(root,artifact,plan,arm,bundle,stop_after=8,deadline_unix=None):
    """Discarded actual full-update calibration; never supplies a quality score."""
    artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True)
    models,initial=initialize_models(arm['config'],artifact)
    model=models[arm['kind']]
    del models
    protocol={'schema':'kernel13-discarded-calibration/v1','arm':arm,
        'source_hashes':stage_definition_hashes(root),'initialization':initial,
        'stop_after':int(stop_after),'quality_selection':'FORBIDDEN','heldout_status':'NOT_RUN'}
    checkpoint=Path(root)/'checkpoints'/plan['run_id']/'calibration'/arm['id']
    result=_run_stage_model(plan,arm['kind'],arm['config'],bundle,artifact,checkpoint,protocol,
        model=model,stop_after=int(stop_after),deadline_unix=deadline_unix)
    if result['resume']['final_step']!=int(stop_after) or any(not r.get('successful',False) for r in result['traces']):
        raise RuntimeError('Calibration must complete all finite updates')
    _release_semantic(arm['config']['device'])
    return result

train_calibration=calibrate_case
